# Nonlinear reaction-diffusion (Jha CMAME), diffusivity field material-property optimization setup

This notebook generates neural-operator training data for the nonlinear reaction-diffusion
forward map $F: M \to U$, matching Jha CMAME's problem setup exactly (see
`chapter_work/applications/topology_optimization_reaction_diffusion`), following the same
data-generation and prior conventions as `survey_work/problems/poisson/Poisson.ipynb`
(same `PriorSampler` structure, same 4000-sample / 3200-train / 800-test convention).
Only DeepONet is trained for this problem's optimization application (it operates directly
on the native unstructured FE mesh), so there is no FNO grid-interpolation step here.
Dependencies are listed in `neuralop.yml` at the repository root.

## Problem description

Domain: $\Omega = (0,1)^2 \setminus \bar B(x_{c1}, R_1) \setminus \bar B(x_{c2}, R_2)$,
$x_{c1}=(0.2,0.8)$, $R_1=0.1$; $x_{c2}=(0.7,0.3)$, $R_2=0.2$ (unit square minus two circular
voids). Let $u: \Omega \to \mathbb{R}$ denote the temperature field:
\begin{equation}\tag{1}
    \begin{aligned}
        -\nabla \cdot \left( m(x) \nabla u(x) \right) + u(x)^3 &= 0\,, \qquad &\forall x \in \Omega\,, \\
        u(x) &= 0\,, \qquad &\forall x \in \Gamma_{in}\ \text{(both void boundaries)}\,, \\
        m(x) \nabla u(x)\cdot n(x) &= g\,, \qquad & \forall x \in \Gamma_{out}\ \text{(outer square boundary)}\,,
    \end{aligned}
\end{equation}
with $g=0.1$ (constant outer flux). This is a NONLINEAR reaction-diffusion equation (the
$u^3$ term), unlike Poisson's linear equation, so $F$ requires a Newton solve rather than a
single linear solve.

The probability distribution for $m$ follows the same construction as Poisson's:
\begin{equation}\tag{2}
    m = \alpha_m \exp(w) + \beta_m, \qquad \text{where} \quad w \sim N(0, C)\,,
\end{equation}
with the SAME covariance-operator parameters as Poisson ($\mathsf{a}_c=0.005$,
$\mathsf{b}_c=1$, $\mathsf{c}_c=0.2$), but $(\alpha_m, \beta_m) = (0.25, 0)$ instead of
Poisson's $(1,0)$ -- matching CMAME's own stated diffusivity parameterization
$m = 0.25\exp(w)$, not an arbitrary choice.

Given $m\in M$, $F(m) = u\in U$ solves the boundary value problem above, i.e.
$F: M \to U$ is the forward operator whose neural-operator surrogate $F_{NOP}$ this notebook's
generated data will train.

## Random samples of m and corresponding solution u(m)

<p align="center"> <img src="./data/ReactionDiffusion_sample_plots.png" width="600"> </p>


In [1]:
import sys
import time
import os

import matplotlib.pyplot as plt
import numpy as np
from dolfinx.fem import functionspace
from mpl_toolkits.axes_grid1 import make_axes_locatable

NOTEBOOK_DIR = os.getcwd()
ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
sys.path.insert(0, os.path.join(ROOT, "src/plotting"))
sys.path.insert(0, os.path.join(ROOT, "src/pde"))
sys.path.insert(0, os.path.join(ROOT, "src/prior"))
sys.path.insert(0, NOTEBOOK_DIR)

from field_plot import field_plot, quick_field_plot
from meshUtilities import get_dirichlet_bc
from plot_mix_collection import plot_collection
from plot_svd import plot_s_vec_values
from mesh_setup import build_mesh, MESH_SIZE
from reactionDiffusionModel import ReactionDiffusionModel
from priorSampler import PriorSampler

plt.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

seed = 0
np.random.seed(seed)

In [2]:
data_folder = "data/"
results_dir = os.path.join(NOTEBOOK_DIR, data_folder)
os.makedirs(results_dir, exist_ok=True)


# Create ReactionDiffusionModel and test prior

In [3]:
def mesh_triangles(domain):
    tdim = domain.topology.dim
    domain.topology.create_connectivity(tdim, 0)
    return domain.topology.connectivity(tdim, 0).array.reshape(-1, 3)

In [4]:
prior_ac = 0.005
prior_cc = 0.2
prior_logn_scale = 0.25   # CMAME's m = 0.25*exp(w), not Poisson's alpha_m=1
prior_logn_translate = 0.0
fe_order = 1
data_prefix = "ReactionDiffusion"

# unstructured gmsh mesh (unit square minus two circular voids) -- NOT a
# structured square mesh like Poisson's, since this domain has holes
mesh_data = build_mesh()
domain = mesh_data.mesh
Vm = functionspace(domain, ("Lagrange", fe_order))
Vu = Vm
elements = mesh_triangles(domain)

prior_sampler = PriorSampler(Vm, prior_ac, prior_cc, seed)
model = ReactionDiffusionModel(
    Vm, Vu, prior_sampler, logn_scale=prior_logn_scale, logn_translate=prior_logn_translate,
    seed=seed,
)
nodes = model.m_nodes

In [5]:
fs = 20
rows, cols = 1, 2
fig, axs = plt.subplots(rows, cols, figsize=(14, 6))
axs = np.atleast_1d(axs).ravel()

for j in range(cols):
    m = (
        prior_sampler.mean
        if j == 0
        else prior_sampler.function_to_vertex(prior_sampler.b_fn, None)
    )
    cbar = field_plot(axs[j], m, nodes, elements=elements, cmap="jet")
    divider = make_axes_locatable(axs[j])
    cax = divider.append_axes("right", size="8%", pad=0.03)
    cax.tick_params(labelsize=fs)
    fig.colorbar(cbar, cax=cax, orientation="vertical")
    axs[j].set_title(r"$\bar{w}$" if j == 0 else r"$\mathsf{b}$", fontsize=fs)
    axs[j].axis("off")

fig.tight_layout()
fig.suptitle(
    r"Prior parameters: Mean ($\bar{w}$) and diffusivity ($\mathsf{b}$)",
    fontsize=1.25 * fs,
    y=1.05,
)
plt.savefig(os.path.join(results_dir, "prior_parameters.png"), bbox_inches="tight")
plt.show()

b_vals = prior_sampler.function_to_vertex(prior_sampler.b_fn, None)
quick_field_plot(
    b_vals,
    nodes,
    title=r"Diffusivity $\mathsf{b}$",
    cmap="jet",
    figsize=(7, 6),
    fs=20,
    savefilename=str(os.path.join(results_dir, "prior_b.png")),
)


/var/folders/tt/zt64sf2n4kgfkx71d8pf39440000gn/T/ipykernel_37685/3702245326.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/prashant/Dropbox/Work/Simulations/neuralnet_works/error_corrector_direction/neural_operators/src/plotting/field_plot.py:230: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
%%time

fs = 20
rows, cols = 3, 3
fig, axs = plt.subplots(rows, cols, figsize=(16, 12))
stats = []
mvec = []
print_info = False
m = prior_sampler.empty_sample()

for i in range(rows):
    for j in range(cols):
        m, log_prior = model.prior_sampler(m)
        mvec.append(m)
        stats.append([np.min(m), np.max(m), np.mean(m), np.std(m), log_prior])
        if print_info:
            print(
                f"Sample: {i * cols + j:2d}, log_prior = {log_prior:.2e}, "
                f"min = {np.min(m):.2e}, max = {np.max(m):.2e}, "
                f"mean = {np.mean(m):.2e}, std = {np.std(m):.2e}"
            )
        cbar = field_plot(axs[i, j], m, model.m_nodes, elements=elements, cmap="jet")
        divider = make_axes_locatable(axs[i, j])
        cax = divider.append_axes("right", size="8%", pad=0.03)
        cax.tick_params(labelsize=fs)
        fig.colorbar(cbar, cax=cax, orientation="vertical")
        axs[i, j].axis("off")

stats = np.array(stats)
print("Statistics of all samples")
print(
    "Mean: min = {:.2e}, max = {:.2e}, mean = {:.2e}, std = {:.2e}, log_prior = {:.2e}".format(
        np.mean(stats[:, 0]),
        np.mean(stats[:, 1]),
        np.mean(stats[:, 2]),
        np.mean(stats[:, 3]),
        np.mean(stats[:, 4]),
    )
)

fig.tight_layout()
fig.suptitle(r"Samples $w\sim N(0,C)$", fontsize=1.25 * fs, y=1.025)
plt.savefig(os.path.join(results_dir, "prior_samples.png"), bbox_inches="tight")
plt.show()

mvec = np.array(mvec)
quick_field_plot(
    np.mean(mvec, axis=0),
    model.m_nodes,
    elements=elements,
    title="Mean of the samples",
    cmap="jet",
)


Statistics of all samples
Mean: min = -5.99e-01, max = 5.41e-01, mean = -1.04e-02, std = 2.26e-01, log_prior = -6.45e-01


CPU times: user 691 ms, sys: 718 ms, total: 1.41 s
Wall time: 470 ms


<timed exec>:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


# Test ReactionDiffusionModel

## Generate few samples of diffusivity and solve the forward problem

In [7]:
n_test_samples = 3

u_vec = []
title_vec = []
cmapvec = []
for i in range(n_test_samples):
    cmapvec.append(["magma", "jet", "jet"])
    if i == 0:
        title_vec.append(
            [r"$w \sim N(0, C)$", r"$m = \alpha_m\, \exp(w) + \beta_m$", r"$u = F(m)$"]
        )
    else:
        title_vec.append([None, None, None])

w = model.empty_m()
m = model.empty_m()
u = model.empty_u()

for i in range(n_test_samples):
    w, log_prior = model.prior_sampler(w)
    m = model.transform_gaussian_pointwise(w, m)
    u = model.solveFwd(u, m, transform_m=False)
    u_vec.append([w, m, u])

plot_collection(
    u_vec,
    n_test_samples,
    3,
    model.m_nodes,
    elements=elements,
    sup_title=r"Samples of $w$ and corresponding $m$ and $u$",
    title_vec=title_vec,
    cmapvec=cmapvec,
    figsize=(16, 12),
    fs=25,
    y_sup_title=1.05,
)


/Users/prashant/Dropbox/Work/Simulations/neuralnet_works/error_corrector_direction/neural_operators/src/plotting/plot_mix_collection.py:255: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
a = model.m_mean
print("stats of mean of m:", a.mean(), a.min(), a.max())
a = model.samplePrior(a, transform_m=True)
print("stats of sample of m:", a.mean(), a.min(), a.max())


stats of mean of m: 0.25 0.25 0.25
stats of sample of m: 0.23358380767189998 0.13596417825239482 0.35484067261351826


## Locate the nodes on the Dirichlet boundary

In [9]:
u_mesh_dirichlet_boundary_nodes = get_dirichlet_bc(
    model.is_point_on_dirichlet_boundary, model.u_nodes
)

print(u_mesh_dirichlet_boundary_nodes.shape, u_mesh_dirichlet_boundary_nodes)

n_test_samples = 10
for i in range(n_test_samples):
    u = model.solveFwd(m=model.samplePrior(transform_m=True), transform_m=False)
    bc_err = np.linalg.norm(u[u_mesh_dirichlet_boundary_nodes])
    print("bc error:", bc_err)


(95,) [ 100  101  119  120  135  136  149  151  164  166  179  180  195  196
  213  214  232  250  259  276  285  300  309  328  338  358  370  388
  399  421  434  455  468  493  506  531  545  571  588  615  662  665
  710  718  757  764  804  812  853  862  897  899  901  909  914  960
  968 1018 1019 1037 1038 1040 1042 2183 2184 2216 2232 2260 2271 2301
 2308 2335 2337 2368 2370 2393 2399 2420 2426 2446 2452 2470 2475 2492
 2496 2509 2521 2527 2540 2545 2557 2558 2567 2568 2570]
bc error: 0.0
bc error: 0.0
bc error: 0.0
bc error: 0.0
bc error: 0.0
bc error: 0.0
bc error: 0.0
bc error: 0.0
bc error: 0.0
bc error: 0.0


## Generate data

Set `generate_data = True` in the cell below to regenerate from scratch (matches the
4000-sample / 3200-train / 800-test convention used in `survey_work/problems/poisson`).
Defaults to loading the existing `.npz` file if set to `False`.


In [10]:
generate_data = True
if generate_data:
    num_samples = 4000

    w_samples = np.zeros((num_samples, model.m_dim))
    m_samples = np.zeros((num_samples, model.m_dim))
    u_samples = np.zeros((num_samples, model.u_dim))

    w, m, u = model.empty_m(), model.empty_m(), model.empty_u()

    for i in range(num_samples):
        start_time = time.perf_counter()

        if i == 0:
            w = prior_sampler.mean.copy()
        else:
            w = prior_sampler(w)[0]

        m = model.transform_gaussian_pointwise(w, m)
        u = model.solveFwd(u, m, transform_m=False)

        w_samples[i, :] = w
        m_samples[i, :] = m
        u_samples[i, :] = u

        if i % 100 == 0:
            print(f"Sample {i:4d} took {time.perf_counter() - start_time:.3f} seconds")

    print(w_samples.shape, m_samples.shape, u_samples.shape)


Sample    0 took 0.010 seconds


Sample  100 took 0.009 seconds


Sample  200 took 0.009 seconds


Sample  300 took 0.009 seconds


Sample  400 took 0.009 seconds


...

Sample 3600 took 0.009 seconds


Sample 3700 took 0.010 seconds


Sample 3800 took 0.009 seconds


Sample 3900 took 0.009 seconds


(4000, 2659) (4000, 2659) (4000, 2659)


In [11]:
plot_from_data = True
if plot_from_data:
    if generate_data is False:
        data_load = np.load(os.path.join(results_dir, f"{data_prefix}_samples.npz"))
        w_samples = data_load["w_samples"]
        m_samples = data_load["m_samples"]
        u_samples = data_load["u_samples"]

    i_choices = np.random.choice(w_samples.shape[0], 3, replace=False)
    n_test_samples = len(i_choices)

    u_vec = []
    title_vec = []
    cmapvec = []
    for i in range(n_test_samples):
        ii = i_choices[i]
        u_vec.append([w_samples[ii, :], m_samples[ii, :], u_samples[ii, :]])
        cmapvec.append(["magma", "jet", "jet"])
        if i == 0:
            title_vec.append(
                [
                    r"$w \sim N(0, C)$",
                    r"$m = \alpha_m\, \exp(w) + \beta_m$",
                    r"$u = F(m)$",
                ]
            )
        else:
            title_vec.append([None, None, None])

    plot_collection(
        u_vec,
        n_test_samples,
        3,
        model.m_nodes,
        elements=elements,
        sup_title=r"Samples of $w$ and corresponding $m$ and $u$",
        title_vec=title_vec,
        cmapvec=cmapvec,
        figsize=(16, 12),
        fs=25,
        y_sup_title=1.05,
        savefilename=str(os.path.join(results_dir, f"{data_prefix}_sample_plots.png")),
    )


In [12]:
if True:
    proj_w_dim, proj_m_dim, proj_u_dim = 100, 100, 100
    tol = 1.0e-9
    if generate_data is False:
        data = np.load(os.path.join(results_dir, f"{data_prefix}_samples.npz"))
        w_samples = data["w_samples"]
        m_samples = data["m_samples"]
        u_samples = data["u_samples"]
        recompute_svd = False
        if recompute_svd:
            w_mean = np.mean(w_samples, axis=0)
            m_mean = np.mean(m_samples, axis=0)
            u_mean = np.mean(u_samples, axis=0)
            w_std = np.std(w_samples, axis=0)
            m_std = np.std(m_samples, axis=0)
            u_std = np.std(u_samples, axis=0)
            w_normalized = (w_samples - w_mean) / (w_std + tol)
            m_normalized = (m_samples - m_mean) / (m_std + tol)
            u_normalized = (u_samples - u_mean) / (u_std + tol)
            w_SVD, w_s, _ = np.linalg.svd(w_normalized.T, full_matrices=False)
            m_SVD, m_s, _ = np.linalg.svd(m_normalized.T, full_matrices=False)
            u_SVD, u_s, _ = np.linalg.svd(u_normalized.T, full_matrices=False)
        else:
            w_SVD = data["w_SVD"]
            m_SVD = data["m_SVD"]
            u_SVD = data["u_SVD"]
            w_s = data["w_s"]
            m_s = data["m_s"]
            u_s = data["u_s"]
    else:
        w_mean = np.mean(w_samples, axis=0)
        m_mean = np.mean(m_samples, axis=0)
        u_mean = np.mean(u_samples, axis=0)
        w_std = np.std(w_samples, axis=0)
        m_std = np.std(m_samples, axis=0)
        u_std = np.std(u_samples, axis=0)
        w_normalized = (w_samples - w_mean) / (w_std + tol)
        m_normalized = (m_samples - m_mean) / (m_std + tol)
        u_normalized = (u_samples - u_mean) / (u_std + tol)
        w_SVD, w_s, _ = np.linalg.svd(w_normalized.T, full_matrices=False)
        m_SVD, m_s, _ = np.linalg.svd(m_normalized.T, full_matrices=False)
        u_SVD, u_s, _ = np.linalg.svd(u_normalized.T, full_matrices=False)


In [13]:
if True:
    plot_annot_xy = [0.15, 0.35, 0.8, 0.5]
    plot_annot_xy_region = [-5, 300, -0.04, 0.15]
    xy_text_vec = []
    xy_text_vec.append([(-30, 30), (10, 10), (-20, -25)])
    xy_text_vec.append([(-10, 15), (5, -5), (-20, -25)])
    xy_text_vec.append([(-40, 15), (-5, 15), (-20, -25)])
    l_style_vec = [":", (0, (5, 10)), "-", "."]

    plot_s_vec_values(
        [w_s, m_s, u_s],
        [proj_w_dim, proj_m_dim, proj_u_dim],
        ["w", "m", "u"],
        l_style_vec,
        xy_text_vec,
        plot_annot_xy,
        plot_annot_xy_region,
        str(os.path.join(results_dir, f"{data_prefix}_svd_analysis_w_m_u")),
    )

    xy_text_vec = []
    xy_text_vec.append([(10, 20), (-20, -25)])
    xy_text_vec.append([(5, -5), (-20, -25)])
    xy_text_vec.append([(-20, 15), (-20, -25)])
    l_style_vec = ["-", "-"]
    plot_s_vec_values(
        [m_s, u_s],
        [proj_m_dim, proj_u_dim],
        ["m", "u"],
        l_style_vec,
        xy_text_vec,
        plot_annot_xy,
        plot_annot_xy_region,
        str(os.path.join(results_dir, f"{data_prefix}_svd_analysis_m_u")),
    )


j = 0, i = 0, index = 100, index_val = 0.028697412899279316
j = 0, i = 1, index = 100, index_val = 0.029428092126137773
j = 0, i = 2, index = 100, index_val = 0.0031989467371206577
j = 1, i = 0, index = 32, index_val = 0.100786363949528
j = 1, i = 1, index = 33, index_val = 0.09923524699380243
j = 1, i = 2, index = 11, index_val = 0.09972950881708771
j = 2, i = 0, index = 256, index_val = 0.009994437041135368
j = 2, i = 1, index = 260, index_val = 0.010012032605025804
j = 2, i = 2, index = 49, index_val = 0.010018480270180046
j = 0, i = 0, index = 100, index_val = 0.029428092126137773
j = 0, i = 1, index = 100, index_val = 0.0031989467371206577
j = 1, i = 0, index = 33, index_val = 0.09923524699380243
j = 1, i = 1, index = 11, index_val = 0.09972950881708771
j = 2, i = 0, index = 260, index_val = 0.010012032605025804
j = 2, i = 1, index = 49, index_val = 0.010018480270180046


/Users/prashant/Dropbox/Work/Simulations/neuralnet_works/error_corrector_direction/neural_operators/src/plotting/plot_svd.py:146: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/prashant/Dropbox/Work/Simulations/neuralnet_works/error_corrector_direction/neural_operators/src/plotting/plot_svd.py:146: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
if generate_data:
    u_mesh_dirichlet_boundary_nodes = get_dirichlet_bc(
        model.is_point_on_dirichlet_boundary, model.u_nodes
    )

    np.savez(
        os.path.join(results_dir, f"{data_prefix}_samples.npz"),
        w_samples=w_samples,
        m_samples=m_samples,
        u_samples=u_samples,
        num_samples=num_samples,
        m_dim=model.m_dim,
        u_dim=model.u_dim,
        fe_order=fe_order,
        mesh_size=MESH_SIZE,
        prior_ac=prior_ac,
        prior_cc=prior_cc,
        prior_alpham=prior_logn_scale,
        prior_betam=prior_logn_translate,
        u_mesh_nodes=model.u_nodes,
        m_mesh_nodes=model.m_nodes,
        u_mesh_elements=elements,
        m_mesh_elements=elements,
        u_mesh_dirichlet_boundary_nodes=u_mesh_dirichlet_boundary_nodes,
        w_SVD=w_SVD,
        w_s=w_s,
        m_SVD=m_SVD,
        m_s=m_s,
        u_SVD=u_SVD,
        u_s=u_s,
    )

    from dolfinx import io

    with io.XDMFFile(domain.comm, str(os.path.join(results_dir, f"{data_prefix}_u_mesh.xdmf")), "w") as xdmf:
        xdmf.write_mesh(domain)
    with io.XDMFFile(domain.comm, str(os.path.join(results_dir, f"{data_prefix}_m_mesh.xdmf")), "w") as xdmf:
        xdmf.write_mesh(domain)

In [15]:
print(
    "m_mesh_nodes shape: {}\nu_mesh_nodes shape: {}\nm_dim: {}\nu_dim: {}".format(
        model.m_nodes.shape, model.u_nodes.shape, model.m_dim, model.u_dim
    )
)


m_mesh_nodes shape: (2659, 2)
u_mesh_nodes shape: (2659, 2)
m_dim: 2659
u_dim: 2659


In [16]:
u_mean = model.solveFwd(u=None, m=model.m_mean, transform_m=False)
quick_field_plot(
    u_mean,
    model.u_nodes,
    elements=elements,
    title=r"$F(\bar{m})$",
    cmap="viridis",
)


/Users/prashant/Dropbox/Work/Simulations/neuralnet_works/error_corrector_direction/neural_operators/src/plotting/field_plot.py:230: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Load data to test

In [17]:
data_load = np.load(os.path.join(results_dir, f"{data_prefix}_samples.npz"))

print(
    "num_samples: {}\nm_samples shape: {}\nu_samples shape: {}\n"
    "m_dim: {}\nu_dim: {}".format(
        data_load["num_samples"],
        data_load["m_samples"].shape,
        data_load["u_samples"].shape,
        data_load["m_dim"],
        data_load["u_dim"],
    )
)


num_samples: 4000
m_samples shape: (4000, 2659)
u_samples shape: (4000, 2659)
m_dim: 2659
u_dim: 2659
